In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf

In [2]:
def read_Bloomberg_port(fn):
    # Read Excel file, skipping the first two rows
    df = pd.read_excel(fn, skiprows=2)
    col_mapping = {
        'Ticker': 'Ticker',
        'Short Name': 'ShortName',
        'PORT US Sz Fact Exp:D-1': 'Size',
        'PORT US Prof Fact Exp:D-1': 'Profitability',
        'PORT US Vol Fact Exp:D-1': 'Volatility',
        'PORT US Trd Act Fact Exp:D-1': 'TradingActivity',
        'PORT US Lev Fact Exp:D-1': 'Leverage',
        'PORT US Mom Fact Exp:D-1': 'Momentum',
        'PORT US Val Fact Exp:D-1': 'Value',
        'PORT US Gr Fact Exp:D-1': 'Growth',
        'PORT US Dvd Yld Fact Exp:D-1': 'DividendYield',
        'PORT US Earn Var Fact Exp:D-1': 'EarningsVariation',
        'P/S': 'PriceToSales',
        'BEst P/S BF12M': 'EstPriceToSales',
        'P/B': 'PriceToBook',
        'BEst P/B BF12M': 'EstPriceToBook',
        'P/E': 'PriceToEarnings',
        'BEst P/E BF12M': 'EstPriceToEarnings',
        'ROE LF': 'ReturnOnEquity',
        'T12M R&D/Sls': 'RnDToSales',
        'GICS Sector': 'Sector',
        'GICS Ind': 'Industry',
        'GICS SubInd': 'SubIndustry',
        'GICS Sector.1': 'SectorName',
        'GICS Ind Name': 'IndustryName',
        'GICS SubInd Name': 'SubIndustryName',
        'ESG Disclosure Score (Latest Available) (BLOOMBERG L.P.)': 'ESGScore',
        'Beta:Y-1': 'Beta',
        'Total Return:Y-1': 'Return1Yr',
        'Number of Employees:Y': 'NumberOfEmployees',
        'Number of Employees:Y-5': 'NumberOfEmployees5YearsAgo'
    }
     # Remove rows with missing "Short Name" and rename columns
    df = df.loc[~df['Short Name'].isna()].rename(columns=col_mapping)
    if 'Ticker.1' in df.columns:
        df.drop(columns=['Ticker.1'], inplace=True)
    # Remove ADRs
    df = df[~df['ShortName'].str.contains('ADR', na=False)]
    # Strip ticker to remove country code or extra parts
    df['Ticker'] = df['Ticker'].str.split().str[0]
    # Keep only tickers with 4 or fewer characters (filter out pink sheets, GDRs, weird share classes)
    df = df[df.Ticker.str.len() <= 4]
    return df

In [3]:
df_2023_tmp = read_Bloomberg_port('data/original_data/20230101_SimplePort.xlsx')
df_2024 = read_Bloomberg_port('data/original_data/20240101_SimplePort.xlsx')
sec2sec_name = df_2023_tmp.set_index('Sector')['SectorName'].to_dict()
secname2sec = df_2023_tmp.set_index('SectorName')['Sector'].to_dict()

In [4]:
# Load historical price data for 2023
prc_df_2023 = pd.read_parquet('data/original_data/prc_df_2023.parquet')

# Calculate daily returns (without automatic filling) and fill remaining NaNs with 0
rets_2023 = prc_df_2023.pct_change(fill_method=None)
rets_2023 = rets_2023.iloc[1:].fillna(0)

# View a sample of returns
rets_2023.head()


Ticker,AAL,AAPL,ABBV,ABNB,ABT,ACGL,ACI,ACN,ADBE,ADI,...,WFC,WM,WMB,WMT,WRB,WTW,XEL,XOM,XYZ,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2023-01-04,0.066719,0.010314,0.008067,0.044994,0.014875,0.004963,0.009662,-0.003404,0.013327,0.021299,...,0.020579,-0.000764,0.006221,0.001114,-0.003307,0.005343,0.008563,0.002911,0.025681,0.014368
2023-01-05,0.029433,-0.010605,-0.001222,-0.011384,-0.003687,0.002708,-0.002392,-0.023613,-0.037990,-0.037490,...,-0.005393,-0.019310,-0.010201,-0.003409,-0.006359,-0.000203,-0.020235,0.022374,-0.023982,-0.023564
2023-01-06,0.013581,0.036794,0.018717,0.009235,0.013809,0.015253,-0.001919,0.023690,0.013123,0.036508,...,0.008958,0.036457,0.016864,0.024499,0.033528,0.028364,0.029607,0.012087,0.066141,0.015057
2023-01-09,0.030324,0.004089,-0.029361,0.008134,-0.001602,-0.019249,0.005286,0.016864,0.027739,0.009546,...,-0.009579,-0.007837,0.002150,-0.012468,-0.017634,0.006432,0.009679,-0.018637,0.001015,-0.003928
2023-01-10,0.039699,0.004456,-0.012495,-0.007844,0.015158,0.006383,0.018165,0.004310,-0.009591,0.012687,...,-0.000708,-0.021044,-0.015017,-0.000621,0.005070,-0.006587,0.001806,0.014935,0.014046,0.049640


In [5]:
# Get valid tickers present in both 2024 features and 2023 price data
valid_tickers = list(set(df_2024['Ticker']).intersection(rets_2023.columns))

# Filter 2023 returns for valid tickers and replace any infinities with 0
returns_clean = rets_2023[valid_tickers].replace([np.inf, -np.inf], 0).fillna(0)

In [6]:
# Calculate a robust covariance matrix using Ledoit-Wolf and annualize it
lw = LedoitWolf().fit(returns_clean)
cov_matrix = pd.DataFrame(lw.covariance_, index=valid_tickers, columns=valid_tickers) * 252

# Set up the optimization problem
n_assets = len(valid_tickers)
initial_weights = np.ones(n_assets) / n_assets

In [7]:
def portfolio_variance(weights):
    return weights.T @ cov_matrix.values @ weights

constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
bounds = [(-0.05, 0.05) for _ in range(n_assets)]

result = minimize(
    portfolio_variance,
    initial_weights,
    method='SLSQP',
    bounds=bounds,
    constraints=constraints,
    options={'maxiter': 1000, 'ftol': 1e-12}
)

if result.success:
    optimized_weights = result.x
else:
    optimized_weights = initial_weights

print("Optimization Status:", "Success" if result.success else "Failed")


Optimization Status: Success


In [8]:
submission_df = pd.DataFrame({'ID': df_2024['Ticker']})

# Map optimized weights (for valid tickers) to the submission DataFrame; tickers not in valid_tickers get weight 0
weight_series = pd.Series(optimized_weights, index=valid_tickers, name='Weights')
submission_df = submission_df.merge(weight_series, how='left', left_on='ID', right_index=True)
submission_df['Weights'] = submission_df['Weights'].fillna(0)

# Enforce weight constraints: clip weights to [-0.05, 0.05] and renormalize so that weights sum to 1 exactly
submission_df['Weights'] = submission_df['Weights'].clip(-0.05, 0.05)
submission_df['Weights'] = submission_df['Weights'] / submission_df['Weights'].sum()

# Save the submission file (CSV with header: ID,WEIGHT)
submission_df.to_csv('data/submission.csv', index=False)
print("Submission file saved as 'submission.csv'")

Submission file saved as 'submission.csv'


In [11]:
submission_df[submission_df["Weights"]==0]

,ID,Weights
25,AEG,0.0
131,CCL,0.0
167,CRBG,0.0
279,GEHC,0.0
351,JBZY,0.0
381,KVUE,0.0
390,LIN,0.0
535,RCI,0.0


In [13]:
weights = np.array(submission_df[submission_df["Weights"]!=0]['Weights'])
portfolio_variance = np.dot(weights.T, np.dot(cov_matrix, weights))
portfolio_variance

0.0030331296703715034